In [7]:
import pandas as pd
from events_data_core.stock_data_provider import get_stock_data

ticker = 'LKOH'
stock_data = get_stock_data(ticker)

stock_data.head()

,DATE,OPEN,HIGH,LOW,CLOSE,VOL
0,2002-01-08,403.15,426.48,403.15,420.00,1385900.0
1,2002-01-09,369.50,433.35,369.50,426.61,1476737.0
2,2002-01-10,424.10,431.30,420.01,426.00,1566808.0
3,2002-01-11,425.00,431.70,423.00,428.01,808508.0
4,2002-01-14,424.90,424.90,415.50,418.00,824284.0


In [8]:
import duckdb

# подключаемся (in-memory, без файла)
con = duckdb.connect()

# читаем два csv
con.execute("""
            CREATE TABLE event_tags AS
            SELECT *
            FROM read_csv_auto('../data/db/event_tags.csv');
            """)

con.execute("""
            CREATE TABLE events AS
            SELECT *
            FROM read_csv_auto('../data/db/events.csv');
            """)

# извлекаем только санкционные события
query = """
        SELECT e.*
        FROM events e
                 JOIN event_tags t ON e.id = t.event_id
        WHERE t.tag_code = 'SANCTIONS'
          and e.date_start > '2022-01-01'
        """

sanctions_df = con.execute(query).df()
sanctions_df

,id,date_start,date_end,event
0,017f1eba-7c00-4bed-8a0c-e5ee2c76f009,2022-02-21,2022-02-23,Принятие первого пакета санкций против России.
1,017f2b9a-7200-4fca-a35a-c68140153c96,2022-02-24,2022-02-25,Принятие второго пакета санкций против России.
2,017f5c86-fc00-47eb-8988-aeb8b647cc14,2022-02-26,2022-03-14,Принятие третьего пакета санкций против России.
3,017fbe5f-7000-4aa7-b4d4-3688dbe73a96,2022-03-15,2022-04-04,Принятие четвертого пакета санкций против России.
4,01808c5d-f000-4f60-85eb-0aa34a373e1a,2022-04-05,2022-06-02,Принятие пятого пакета санкций против России.
5,01819302-7400-4bdb-8219-df4aec4b9fc7,2022-06-03,2022-07-15,Принятие шестого пакета санкций против России.
6,0182df2c-7200-4a21-96e5-597baa799d81,2022-07-21,2022-10-04,Принятие седьмого пакета санкций против России.
7,01845ed6-7800-41fd-af56-a56104072a56,2022-10-06,2022-12-15,Принятие 8 пакета санкций против России.
8,0185cc79-fc00-46ca-8a45-fd3c3a0342f8,2022-12-16,2023-02-24,Принят 9 пакет санкций против России.
9,0187b08f-7400-42c0-86b2-cb3b9c32c53a,2023-02-25,2023-06-21,Принятие 10 пакета санкций против России.


In [9]:
from events_data_core.plot_utils import plot_2d_events

app = plot_2d_events(stock_data['DATE'], stock_data['CLOSE'], sanctions_df)

app.run(debug=True, jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/


In [10]:

import numpy as np
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from events_data_core.cpi_data_provider import load_normalized_ipc_data, IpcType

pd.set_option('future.no_silent_downcasting', True)

# ── 1. Нормализация по CPI (forward fill, база = 2022-01-01) ──────────────
BASE_DATE = '2022-01-01'
cpi_norm = load_normalized_ipc_data(IpcType.GOODS_AND_SERVICES, base_year=BASE_DATE)
cpi_norm['date'] = pd.to_datetime(cpi_norm['date'])

stock = stock_data.copy()
stock['DATE'] = pd.to_datetime(stock['DATE'])
stock = stock.sort_values('DATE').reset_index(drop=True)

# forward fill: каждому торговому дню присваивается real_ruble за его месяц
stock = pd.merge_asof(
    stock, cpi_norm[['date', 'real_ruble']],
    left_on='DATE', right_on='date', direction='backward'
)
stock['CLOSE_REAL'] = stock['CLOSE'] * stock['real_ruble']

# ── 2. Параметры ──────────────────────────────────────────────────────────
WINDOW          = 20            # торговых дней (регулируемый параметр)
BASELINE_START  = '2020-07-01' # начало довоенного baseline (после ковид-восстановления)
BASELINE_END    = '2021-12-31' # конец довоенного baseline

print(f"[Параметры] window={WINDOW} торг. дней | CPI база={BASE_DATE} | "
      f"baseline={BASELINE_START} – {BASELINE_END}")

# ── 3. Индексируем price и volume по дате ─────────────────────────────────
price       = stock.set_index('DATE')['CLOSE_REAL']
vol_series  = stock.set_index('DATE')['VOL']
t_days      = price.index.sort_values()
event_dates = pd.to_datetime(sanctions_df['date_start'].values)

# Фиксированный довоенный baseline для CAR (одинаковый для всех событий)
baseline_idx     = t_days[(t_days >= BASELINE_START) & (t_days <= BASELINE_END)]
baseline_returns = price.reindex(baseline_idx).dropna().pct_change().dropna()
baseline_daily   = baseline_returns.mean()

print(f"Baseline: {len(baseline_idx)} торг. дней, "
      f"средняя дневная доходность = {baseline_daily * 100:.4f}%")

# ── 4. Расчёт метрик для каждого пакета санкций ───────────────────────────
records = []

for i, (_, row) in enumerate(sanctions_df.iterrows()):
    ev_date = pd.Timestamp(row['date_start'])

    # Границы от соседних событий (половина расстояния до соседа)
    prev_events = event_dates[event_dates < ev_date]
    next_events = event_dates[event_dates > ev_date]
    prev_bound  = prev_events.max() if len(prev_events) > 0 else ev_date - pd.Timedelta(days=9999)
    next_bound  = next_events.min() if len(next_events) > 0 else ev_date + pd.Timedelta(days=9999)

    max_before_date = prev_bound + (ev_date - prev_bound) / 2
    max_after_date  = ev_date   + (next_bound - ev_date)  / 2

    # Торговые дни в окнах (с обрезанием)
    before_all = t_days[(t_days < ev_date)  & (t_days >= max_before_date)][-WINDOW:]
    after_all  = t_days[(t_days > ev_date)  & (t_days <= max_after_date)][:WINDOW]

    clipped = len(before_all) < WINDOW or len(after_all) < WINDOW

    p_before = price.reindex(before_all).dropna()
    p_after  = price.reindex(after_all).dropna()
    v_before = vol_series.reindex(before_all).dropna()
    v_after  = vol_series.reindex(after_all).dropna()

    if len(p_before) < 2 or len(p_after) < 2:
        continue

    # % Доходность точечная: price[last after] vs price[first before]
    ret_point = (p_after.iloc[-1] - p_before.iloc[0]) / p_before.iloc[0] * 100

    # % Доходность средняя: mean(after) vs mean(before)
    ret_avg = (p_after.mean() - p_before.mean()) / p_before.mean() * 100

    # CAR: baseline из довоенного периода, аномальная доходность в after
    abnormal = p_after.pct_change().dropna() - baseline_daily
    car      = abnormal.sum() * 100

    # Волатильность (std дневных доходностей)
    vb      = p_before.pct_change().dropna().std() * 100
    va      = p_after.pct_change().dropna().std()  * 100
    v_ratio = round(va / vb, 2) if vb > 0 else np.nan

    # Объём
    vol_b     = v_before.mean()
    vol_a     = v_after.mean()
    vol_ratio = round(vol_a / vol_b, 2) if vol_b > 0 else np.nan

    records.append({
        'Пакет':                  f"Пакет {i + 1}",
        'Дата':                   ev_date.date(),
        'Обрезано':               clipped,
        'Дней до':                len(p_before),
        'Дней после':             len(p_after),
        'Доходность точечная, %': round(ret_point, 2),
        'Доходность средняя, %':  round(ret_avg, 2),
        'CAR, %':                 round(car, 2),
        'Волатильность до, %':    round(vb, 3),
        'Волатильность после, %': round(va, 3),
        'Коэф. волатильности':    v_ratio,
        'Объём до':               int(vol_b),
        'Объём после':            int(vol_a),
        'Коэф. объёма':           vol_ratio,
    })

results_df = pd.DataFrame(records)
display(results_df)

# ── 5. Графики ────────────────────────────────────────────────────────────
labels     = results_df['Пакет']
colors_ret = ['green' if v >= 0 else 'red' for v in results_df['Доходность средняя, %']]
colors_car = ['green' if v >= 0 else 'red' for v in results_df['CAR, %']]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Средняя доходность, %',
        'CAR (накопл. аномальная доходность), %',
        'Коэффициент волатильности (после / до)',
        'Коэффициент объёма торгов (после / до)',
    ]
)

fig.add_trace(go.Bar(x=labels, y=results_df['Доходность средняя, %'],  marker_color=colors_ret,   name='Доходность'),    row=1, col=1)
fig.add_trace(go.Bar(x=labels, y=results_df['CAR, %'],                  marker_color=colors_car,   name='CAR'),           row=1, col=2)
fig.add_trace(go.Bar(x=labels, y=results_df['Коэф. волатильности'],     marker_color='steelblue',  name='Волатильность'), row=2, col=1)
fig.add_trace(go.Bar(x=labels, y=results_df['Коэф. объёма'],            marker_color='darkorange', name='Объём'),         row=2, col=2)

fig.add_hline(y=1, row=2, col=1, line_dash='dash', line_color='gray', line_width=1)
fig.add_hline(y=1, row=2, col=2, line_dash='dash', line_color='gray', line_width=1)

fig.update_layout(
    height=750,
    title_text=(
        f'Влияние пакетов санкций на акции LKOH  |  window={WINDOW} дн.  |  '
        f'CPI база={BASE_DATE}  |  baseline={BASELINE_START}–{BASELINE_END}'
    ),
    showlegend=False,
    template='plotly_white',
)

out_path = os.path.abspath('topic_plots/sanctions_impact.html')
fig.write_html(out_path)
print(f"График готов: {out_path}")


[Параметры] window=20 торг. дней | CPI база=2022-01-01 | baseline=2020-07-01 – 2021-12-31
Baseline: 384 торг. дней, средняя дневная доходность = 0.0389%


,Пакет,Дата,Обрезано,Дней до,Дней после,"Доходность точечная, %","Доходность средняя, %","CAR, %","Волатильность до, %","Волатильность после, %",Коэф. волатильности,Объём до,Объём после,Коэф. объёма
0,Пакет 5,2022-04-05,True,6,19,-12.53,-13.56,-13.55,5.973,4.598,0.77,386272,588186,1.52
1,Пакет 6,2022-06-03,True,19,15,-11.99,-5.69,3.81,2.765,2.390,0.86,396115,496259,1.25
2,Пакет 7,2022-07-21,True,18,20,-3.51,0.52,6.24,1.654,1.964,1.19,456949,459761,1.01
3,Пакет 8,2022-10-06,False,20,20,5.70,4.87,18.26,3.192,1.681,0.53,830156,785829,0.95
4,Пакет 9,2022-12-16,False,20,20,-14.33,-11.97,-14.68,0.657,2.747,4.18,405170,474381,1.17
5,Пакет 10,2023-02-25,False,20,20,9.43,4.50,9.26,0.818,1.264,1.54,380287,609003,1.60
6,Пакет 11,2023-06-23,False,20,20,4.74,0.64,7.43,2.251,1.152,0.51,1309456,943180,0.72
7,Пакет 12,2023-12-18,False,20,20,-5.46,-5.17,1.40,1.473,0.712,0.48,712056,484527,0.68
8,Пакет 13,2024-02-23,False,20,20,6.64,3.20,2.71,0.823,0.983,1.19,576267,1057611,1.84
9,Пакет 14,2024-06-24,False,20,20,-10.82,-5.02,-4.36,1.910,1.716,0.90,1096880,796125,0.73


График готов: C:\Users\Ruslan\PycharmProjects\data-core\events_data_core\topic_plots\sanctions_impact.html


In [12]:

# ── Реальная цена LKOH, нормализованная к 1-му дню войны (24.02.2022) ────
WAR_START = pd.Timestamp('2022-02-24')

# ближайший торговый день к дате начала войны
war_idx   = t_days[t_days >= WAR_START][0]
war_price = price.loc[war_idx]

price_norm = (price / war_price * 100).reset_index()
price_norm.columns = ['date', 'price_norm']

# только с 2021-01-01, чтобы был контекст до войны
price_norm = price_norm[price_norm['date'] >= '2021-01-01']

fig2 = go.Figure()

# основная линия
fig2.add_trace(go.Scatter(
    x=price_norm['date'],
    y=price_norm['price_norm'],
    mode='lines',
    line=dict(color='steelblue', width=2),
    name='Реальная цена LKOH',
    hovertemplate='%{x|%d.%m.%Y}: %{y:.1f}<extra></extra>',
))

# горизонтальная линия = 100 (уровень 24.02.2022)
fig2.add_hline(y=100, line_dash='dash', line_color='red', line_width=1.5,
               annotation_text='24.02.2022 = 100', annotation_position='top left')

# вертикальные линии для каждого пакета санкций
for _, row in sanctions_df.iterrows():
    ev = pd.Timestamp(row['date_start'])
    if ev >= price_norm['date'].min():
        fig2.add_vline(x=ev, line_width=1, line_dash='dot', line_color='rgba(200,50,50,0.5)')

fig2.update_layout(
    title='Реальная цена акций LKOH (с поправкой на инфляцию), нормализована к 24.02.2022 = 100',
    xaxis_title='Дата',
    yaxis_title='Индекс (24.02.2022 = 100)',
    template='plotly_white',
    height=500,
    hovermode='x unified',
)

out_path2 = os.path.abspath('topic_plots/lkoh_real_price_normalized.html')
fig2.write_html(out_path2)
print(f"График готов: {out_path2}")


График готов: C:\Users\Ruslan\PycharmProjects\data-core\events_data_core\topic_plots\lkoh_real_price_normalized.html
